In [2]:
import os
import time
from pathlib import Path
from dotenv import load_dotenv
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import save_output
from marker.config.parser import ConfigParser
from marker.util import strings_to_classes

In [4]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
#OPENAI_API_KEY = os.getenv("ANDRES_API_KEY")
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
# 2. Configuración de rutas
#input_dir = Path("./Marleth_PDFs")  # Carpeta con tus PDFs
input_dir = Path("./publications/pdf_papers")  # Carpeta con tus PDFs
#output_base_dir = Path("./Marleth_PDFs/markdowns")
output_base_dir = Path("./publications/output_marker/LLM_gpt-4o-mini")
output_base_dir.mkdir(parents=True, exist_ok=True)

In [6]:
# 1. Crear el diccionario de modelos 
model_dict = create_model_dict()

2026-02-03 12:01:10,055 [WARNING] surya: `TableRecEncoderDecoderModel` is not compatible with mps backend. Defaulting to cpu instead


In [7]:
# Verificar en qué dispositivo quedó cada modelo
for model_name, model_obj in model_dict.items():
    try:
        # Algunos modelos son objetos complejos, intentamos ver el device del primer parámetro
        device = next(model_obj.model.parameters()).device
        print(f"✅ Modelo {model_name} cargado en: {device}")
    except:
        print(f"ℹ️ Modelo {model_name} (estructura compleja/CPU fallback)")

ℹ️ Modelo layout_model (estructura compleja/CPU fallback)
ℹ️ Modelo recognition_model (estructura compleja/CPU fallback)
✅ Modelo table_rec_model cargado en: cpu
✅ Modelo detection_model cargado en: mps:0
✅ Modelo ocr_error_model cargado en: mps:0


In [8]:
def process_papers(use_llm=False, force_ocr=False, model_name="gpt-4o-mini"):
    abs_input_dir = input_dir.resolve()
    pdf_files = list(abs_input_dir.glob("*.pdf"))
    
    for pdf_path in pdf_files:
        paper_id = pdf_path.stem
        paper_output_dir = output_base_dir / paper_id

        if paper_output_dir.exists():
            print(f"⏭️  Saltando {paper_id}, ya existe la carpeta.")
            continue

        print(f"📄 Procesando: {paper_id} (LLM: {use_llm}, OCR: {force_ocr})...")

        try:
            paper_output_dir.mkdir(parents=True, exist_ok=True)

            time_start = time.time()

            # --- 1. Config común (controla OCR, LLM, etc.) ---
            master_config = {
                "output_format": "markdown",  # el renderer inicial da igual, usaremos build_document
                "use_llm": use_llm,
                "llm_service": "marker.services.openai.OpenAIService" if use_llm else None,
                "openai_api_key": OPENAI_API_KEY if use_llm else None,
                "openai_model": model_name if use_llm else None,
                "force_ocr": force_ocr,
                "langs": ["en"],
            }

            parser_master = ConfigParser(master_config)

            # Creamos UN PdfConverter por PDF
            converter = PdfConverter(
                config=parser_master.generate_config_dict(),
                artifact_dict=model_dict,
                processor_list=parser_master.get_processors(),
                renderer=None,  # no usamos __call__, así que da igual
                llm_service=parser_master.get_llm_service() if use_llm else None,
            )

            # --- 2. UNA sola pasada pesada: build_document ---
            print("  🚀 Construyendo Document (OCR + layout + LLM + procesadores)...")
            document = converter.build_document(str(pdf_path.absolute()))
            print(f"  ✅ Document listo con {len(document.pages)} páginas.")

            # --- 3. Función auxiliar para renderizar un formato desde el mismo Document ---
            def render_format(fmt: str, suffix: str):
                fmt_dir = paper_output_dir / fmt
                fmt_dir.mkdir(parents=True, exist_ok=True)

                fmt_config = {
                    "output_format": fmt,
                    "langs": ["en"],
                }
                parser_fmt = ConfigParser(fmt_config)

                # Nombre de la clase de renderer (string)
                renderer_name = parser_fmt.get_renderer()

                # Lo convertimos a clase
                renderer_cls = strings_to_classes([renderer_name])[0]

                # Instanciamos el renderer con las dependencias necesarias
                renderer = converter.resolve_dependencies(renderer_cls)

                # Renderizamos desde el mismo Document (no se llama build_document aquí)
                rendered = renderer(document)

                save_output(rendered, str(fmt_dir.absolute()), f"{paper_id}_{suffix}")

            # a) Markdown
            print("  ⚡ Renderizando markdown...")
            render_format("markdown", "md")

            # b) JSON
            print("  ⚡ Renderizando JSON...")
            render_format("json", "json")

            # c) Chunks (si los quieres)
            print("  ⚡ Renderizando chunks...")
            render_format("chunks", "chunks")

            time_end = time.time()

            print(f"✅ Finalizado con éxito: {paper_id} en {time_end - time_start:.2f} segundos.\n")

            # Crear log de tiempos de procesamiento
            csv_path = output_base_dir / "processing_times.csv"
            write_header = not os.path.exists(csv_path)

            with open(csv_path, "a") as f:
                if write_header:
                    f.write("paper_id,num_pages,processing_time_seconds\n")
                f.write(f"{paper_id},{len(document.pages)},{time_end - time_start:.2f}\n")

        except Exception as e:
            print(f"❌ Error en {paper_id}: {str(e)}")


In [10]:
#process_papers(use_llm=True, force_ocr=False, model_name="gpt-4o-mini")
process_papers(use_llm=True, force_ocr=False, model_name="gpt-5-mini")

⏭️  Saltando 216, ya existe la carpeta.
⏭️  Saltando 160, ya existe la carpeta.
⏭️  Saltando 174, ya existe la carpeta.
⏭️  Saltando 49, ya existe la carpeta.
⏭️  Saltando 61, ya existe la carpeta.
⏭️  Saltando 75, ya existe la carpeta.
⏭️  Saltando 60, ya existe la carpeta.
⏭️  Saltando 149, ya existe la carpeta.
⏭️  Saltando 175, ya existe la carpeta.
⏭️  Saltando 161, ya existe la carpeta.
⏭️  Saltando 203, ya existe la carpeta.
⏭️  Saltando 217, ya existe la carpeta.
⏭️  Saltando 201, ya existe la carpeta.
⏭️  Saltando 177, ya existe la carpeta.
⏭️  Saltando 89, ya existe la carpeta.
⏭️  Saltando 188, ya existe la carpeta.
⏭️  Saltando 76, ya existe la carpeta.
⏭️  Saltando 62, ya existe la carpeta.
⏭️  Saltando 189, ya existe la carpeta.
⏭️  Saltando 162, ya existe la carpeta.
⏭️  Saltando 176, ya existe la carpeta.
⏭️  Saltando 88, ya existe la carpeta.
⏭️  Saltando 238, ya existe la carpeta.
⏭️  Saltando 204, ya existe la carpeta.
⏭️  Saltando 210, ya existe la carpeta.
⏭️  Salt

LLMTableProcessor running:  25%|██▌       | 2/8 [00:56<03:00, 30.16s/it]2026-02-03 13:29:57,856 [WARNING] marker: Rate limit error: Request timed out.. Retrying in 3 seconds... (Attempt 1/3)
2026-02-03 13:30:16,677 [WARNING] marker: Rate limit error: Request timed out.. Retrying in 3 seconds... (Attempt 1/3)
2026-02-03 13:30:54,931 [WARNING] marker: Rate limit error: Request timed out.. Retrying in 3 seconds... (Attempt 1/3)
2026-02-03 13:31:32,434 [WARNING] marker: Rate limit error: Request timed out.. Retrying in 6 seconds... (Attempt 2/3)
2026-02-03 13:31:51,298 [WARNING] marker: Rate limit error: Request timed out.. Retrying in 6 seconds... (Attempt 2/3)
2026-02-03 13:32:29,761 [WARNING] marker: Rate limit error: Request timed out.. Retrying in 6 seconds... (Attempt 2/3)
2026-02-03 13:33:09,900 [ERROR] marker: Rate limit error: Request timed out.. Max retries reached. Giving up. (Attempt 3/3)
LLMTableProcessor running:  62%|██████▎   | 5/8 [05:41<03:15, 65.16s/it]2026-02-03 13:34:4

  ✅ Document listo con 23 páginas.
  ⚡ Renderizando markdown...
  ⚡ Renderizando JSON...
  ⚡ Renderizando chunks...
✅ Finalizado con éxito: 33 en 2916.74 segundos.

⏭️  Saltando 27, ya existe la carpeta.
⏭️  Saltando 132, ya existe la carpeta.
⏭️  Saltando 293, ya existe la carpeta.
⏭️  Saltando 244, ya existe la carpeta.
⏭️  Saltando 250, ya existe la carpeta.
⏭️  Saltando 278, ya existe la carpeta.
⏭️  Saltando 237, ya existe la carpeta.
⏭️  Saltando 141, ya existe la carpeta.
⏭️  Saltando 155, ya existe la carpeta.
⏭️  Saltando 97, ya existe la carpeta.
⏭️  Saltando 169, ya existe la carpeta.
⏭️  Saltando 182, ya existe la carpeta.
⏭️  Saltando 196, ya existe la carpeta.
⏭️  Saltando 54, ya existe la carpeta.
⏭️  Saltando 7, ya existe la carpeta.
⏭️  Saltando 69, ya existe la carpeta.
⏭️  Saltando 197, ya existe la carpeta.
⏭️  Saltando 183, ya existe la carpeta.
⏭️  Saltando 96, ya existe la carpeta.
⏭️  Saltando 168, ya existe la carpeta.
⏭️  Saltando 82, ya existe la carpeta.
⏭️ 